In [9]:
import torch
import torch.nn as nn

In [10]:
#Quy trình hoàn chỉnh
class TransformerToyModel(nn.Module):
    def __init__(self, vocab_size, d_model=512):
        super().__init__()
        #Khởi tạo Embedding (Mã hóa ý nghĩa và vị trí)
        self.embedding = nn.Embedding(vocab_size, d_model)
        #Giả xử câu tối đa 100 từ
        self.pos_encoding = nn.Parameter(torch.randn(1, 100, d_model))
        
        #Khởi tạo Encoder Block
        self.encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=8, dim_feedforward=2048, batch_first=True)
        #Khởi tạo Decoder Block [6, 7]
        self.decoder_layer = nn.TransformerDecoderLayer(d_model=d_model, nhead=8, dim_feedforward=2048, batch_first=True)
        
        #Prediction Head (Chuyển về xác suất từ vựng)
        self.prediction_head = nn.Linear(d_model, vocab_size)

    def forward(self, src, tgt):
        #Embedding cho chuỗi nguồn và chuỗi đích
        src_emb = self.embedding(src) + self.pos_encoding[:, :src.size(1), :]
        tgt_emb = self.embedding(tgt) + self.pos_encoding[:, :tgt.size(1), :]
        
        #Encoder xử lý chuỗi nguồn để tạo ngữ cảnh phong phú
        enc_output = self.encoder_layer(src_emb)
        
        #Decoder nhận thông tin từ Encoder và chuỗi đích hiện tại
        #dec_output = decoder(target, memory/encoder_output)
        dec_output = self.decoder_layer(tgt_emb, enc_output)
        
        #Dự đoán xác suất cho token kế tiếp
        logits = self.prediction_head(dec_output)
        return logits

In [11]:
#Cấu hình Demo
vocab_size = 37000 #Kích thước bộ từ vựng theo bài báo gốc
d_model = 512
model = TransformerToyModel(vocab_size, d_model)

#Tạo Input thực tế (Không để tensor rỗng)
#Giả sử Batch size = 1, Câu nguồn có 7 từ, Câu đích hiện tại có 3 từ
src_input = torch.randint(0, vocab_size, (1, 7)) 
tgt_input = torch.randint(0, vocab_size, (1, 3)) 

print("BẮT ĐẦU CHẠY TRANSFORMER")
try:
    #Thực hiện forward pass
    output_logits = model(src_input, tgt_input)
    
    #Lấy dự đoán cho token cuối cùng
    next_token_id = torch.argmax(output_logits[:, -1, :], dim=-1)

    print(f"Kích thước Input nguồn: {src_input.shape}")
    print(f"Kích thước Output Logits: {output_logits.shape}") # (1, 3, 37000)
    print(f"ID của từ tiếp theo được AI dự đoán: {next_token_id.item()}")
except Exception as e:
    print(f"Error: {e}")

BẮT ĐẦU CHẠY TRANSFORMER
Kích thước Input nguồn: torch.Size([1, 7])
Kích thước Output Logits: torch.Size([1, 3, 37000])
ID của từ tiếp theo được AI dự đoán: 29631
